In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Silver_Payments
# Bronze -> Silver
# ============================================================

SOURCE_TABLE = "bronze_payments"
ORDERS_TABLE = "silver_orders"
SILVER_TABLE = "silver_payments"
QUARANTINE_TABLE = "quarantine_payments"

# ------------------------------------------------------------
# 1. Read Bronze
# ------------------------------------------------------------

payments = spark.table(SOURCE_TABLE)

orders = (
    spark.table(ORDERS_TABLE)
    .select("order_id")
    .dropDuplicates()
)

# ------------------------------------------------------------
# 2. Standardize types
# ------------------------------------------------------------

payments = (
    payments
    .withColumn("payment_id", F.col("payment_id").cast("int"))
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("payment_date", F.to_timestamp("payment_date"))
    .withColumn(
        "payment_method",
        F.lower(F.trim(F.col("payment_method").cast("string")))
    )
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .withColumn(
        "status",
        F.lower(F.trim(F.col("status").cast("string")))
    )
)

# ------------------------------------------------------------
# 3. Referential integrity
# ------------------------------------------------------------

payments = (
    payments
    .join(
        orders.withColumn("order_exists", F.lit(True)),
        on="order_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 4. Validation
# ------------------------------------------------------------

df_checked = (
    payments
    .withColumn(
        "dq_reason",
        F.when(
            F.col("payment_id").isNull(),
            F.lit("NULL_PAYMENT_ID")
        )
        .when(
            F.col("order_id").isNull(),
            F.lit("NULL_ORDER_ID")
        )
        .when(
            F.col("order_exists").isNull(),
            F.lit("INVALID_ORDER_REFERENCE")
        )
        .when(
            F.col("payment_date").isNull(),
            F.lit("NULL_PAYMENT_DATE")
        )
        .when(
            F.col("payment_method").isNull() |
            (F.col("payment_method") == ""),
            F.lit("NULL_OR_EMPTY_PAYMENT_METHOD")
        )
        .when(
            F.col("amount").isNull(),
            F.lit("NULL_AMOUNT")
        )
        .when(
            F.col("amount") < 0,
            F.lit("NEGATIVE_PAYMENT_AMOUNT")
        )
        .when(
            F.col("status").isNull() |
            (F.col("status") == ""),
            F.lit("NULL_OR_EMPTY_STATUS")
        )
        .otherwise(F.lit(None))
    )
)

# ------------------------------------------------------------
# 5. Quarantine
# ------------------------------------------------------------

df_quarantine = (
    df_checked
    .filter(F.col("dq_reason").isNotNull())
    .drop("order_exists")
    .withColumn("quarantined_at", F.current_timestamp())
)

df_quarantine.write.mode("overwrite").format("delta").saveAsTable(
    QUARANTINE_TABLE
)

# ------------------------------------------------------------
# 6. Valid records
# ------------------------------------------------------------

df_valid = (
    df_checked
    .filter(F.col("dq_reason").isNull())
    .drop("dq_reason", "order_exists")
)

# ------------------------------------------------------------
# 7. Deduplicate
# ------------------------------------------------------------

window = Window.partitionBy("payment_id").orderBy(
    F.col("payment_date").desc_nulls_last()
)

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# ------------------------------------------------------------
# 8. Write Silver
# ------------------------------------------------------------

df_dedup.write.mode("overwrite").format("delta").saveAsTable(
    SILVER_TABLE
)

print(f"Silver table created: {SILVER_TABLE}")
print(f"Valid records: {df_dedup.count()}")
print(f"Quarantined records: {df_quarantine.count()}")

StatementMeta(, 86e0f46e-6934-4a9b-9f0b-8bdee0a17eb0, 3, Finished, Available, Finished, False)

Silver table created: silver_payments
Valid records: 49978
Quarantined records: 22
